In [1]:
import os
import cv2
import numpy as np
from collections import Counter

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

In [2]:

DATASET_PATH = r"C:\capstone project\MEDDS\Face_Model\DataSet_Face"

#  Label mapping
label_map = {
    "angry": 0,
    "disgust": 1,
    "fear": 2,
    "happy": 3,
    "sad": 4,
    "surprise": 5,
    "neutral": 6
}

IMG_SIZE = 96

X = []
y = []

# 🔄 Load and clean dataset
for label_name in os.listdir(DATASET_PATH):
    if label_name not in label_map:
        continue

    label = label_map[label_name]
    folder_path = os.path.join(DATASET_PATH, label_name)

    for file in os.listdir(folder_path):
        img_path = os.path.join(folder_path, file)

        img = cv2.imread(img_path)

        # ❌ skip bad images
        if img is None:
            continue

        try:
            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
            img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
            img = img / 255.0

            X.append(img)
            y.append(label)
        except:
            continue


In [ ]:



# 📏 Convert to numpy
X = np.array(X).reshape(-1, IMG_SIZE, IMG_SIZE, 1)
y = np.array(y)

print("Dataset shape:", X.shape)
print("Label distribution:", Counter(y))

# 🧠 Build CNN model with Softmax
model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(IMG_SIZE, IMG_SIZE, 1)),
    MaxPooling2D(2,2),

    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2,2),

    Conv2D(128, (3,3), activation='relu'),
    MaxPooling2D(2,2),

    Flatten(),

    Dense(128, activation='relu'),
    Dropout(0.5),

    Dense(7, activation='softmax')  # 🔥 Softmax layer
])

# ⚙️ Compile
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# 🧪 Train
model.fit(X, y, epochs=15, batch_size=32, validation_split=0.2)

# 💾 Save model
model.save("emotion_model.h5")

print("✅ Training complete. Model saved as emotion_model.h5")

Dataset shape: (78488, 96, 96, 1)
Label distribution: Counter({np.int64(3): 18613, np.int64(6): 13131, np.int64(4): 11365, np.int64(2): 10017, np.int64(0): 9915, np.int64(5): 9091, np.int64(1): 6356})


c:\capstone project\MEDDS\.venv\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/15
